# Dónde construir: estaciones para una alcaldía nueva

## Contexto

Una nueva administración acaba de asumir la Alcaldía de Medellín. Dentro del plan de desarrollo se aprobó una partida para construir estaciones que atiendan tres tipos de incidentes:

| Tipo de incidente | Estación que lo atiende |
|---|---|
| Hurto | CAI de policía |
| Accidente de tránsito | Estación de tránsito y grúas |
| Incendio o emergencia | Estación de bomberos |

La Secretaría de Planeación entrega un histórico de incidentes de los últimos años, la red vial de la ciudad y una lista de lotes disponibles donde se podría construir. Cada estación tiene un costo que depende de su tipo y de la zona del lote, y solo atiende con oportunidad los incidentes de su tipo que estén a cierta distancia por las calles.

El presupuesto no alcanza para cubrir toda la ciudad. Tu equipo debe decidir **qué estaciones construir y en qué lotes**, de manera que la ciudad quede lo mejor atendida posible sin exceder el presupuesto, y presentar la propuesta al concejo con un mapa, el costo y los casos que se cubrirían.

La red vial corresponde a las calles reales de Medellín. El histórico de incidentes, los lotes y los costos son simulados para el curso.

## Los datos

Cada instancia está en una carpeta con cinco archivos:

- **`nodos.csv`**: intersecciones y puntos de la red. Columnas `id`, `lat`, `lon`.
- **`aristas.csv`**: tramos de calle. Columnas `origen`, `destino`, `longitud_m`, `nombre`. Cada fila es un tramo que se puede recorrer **solo de `origen` a `destino`**. Una calle de doble sentido aparece dos veces, una por cada dirección. Algunos tramos no tienen nombre.
- **`lotes.csv`**: lotes disponibles. Columnas `lote` (código), `nodo` (intersección donde está el lote) y `factor_zona`.
- **`incidentes.csv`**: histórico. Columnas `id`, `nodo` (intersección donde ocurrió), `tipo` (`hurto`, `accidente` o `incendio`) y `fecha_hora`.
- **`parametros.json`**: presupuesto total y, para cada tipo de incidente, el nombre de la estación, su costo base, su radio de cobertura y su peso de prioridad.

Los parámetros de las tres instancias son los mismos salvo el presupuesto:

| Tipo | Estación | Costo base (millones) | Radio de cobertura | Peso |
|---|---|---|---|---|
| hurto | CAI | 1.000 | 1.000 m | 1 |
| accidente | Tránsito y grúas | 2.000 | 2.000 m | 3 |
| incendio | Bomberos | 5.000 | 3.000 m | 5 |

### Las tres instancias

| Instancia | Descripción | Nodos | Tramos | Incidentes | Lotes | Presupuesto (millones) |
|---|---|---|---|---|---|---|
| `pequena` | Barrio ficticio | 9 | 22 | 18 | 5 | 6.000 |
| `comuna` | Recorte de la red real alrededor de Laureles | 2.893 | 6.782 | 913 | 15 | 12.000 |
| `ciudad` | Red vial completa de Medellín | 27.568 | 67.330 | 5.300 | 80 | 40.000 |

### Descripción del barrio de la instancia `pequena`

El barrio es una cuadrícula de 3 × 3 intersecciones numeradas de 0 a 8. Vista como un mapa con el norte hacia arriba:

```
 6 ── 7 ── 8        fila de arriba:  Calle 3
 │    │    ↑
 3 ── 4 →─ 5        fila del medio:  Calle 2
 │    │    │
 0 ── 1 ── 2        fila de abajo:   Calle 1

Carrera A (0-3-6)   Carrera B (1-4-7)   Carrera C (2-5-8)
```

Todas las calles son de doble sentido excepto dos tramos: en la Calle 2 solo se puede ir **de 4 a 5**, y en la Carrera C solo se puede ir **de 5 a 8**. Cada tramo mide entre 900 m y 990 m; las longitudes exactas están en `aristas.csv`.

Los lotes disponibles están en las intersecciones 1, 3, 5, 6 y 8:

| lote | nodo | factor_zona |
|---|---|---|
| L1 | 1 | 1.1 |
| L2 | 3 | 1.5 |
| L3 | 5 | 1.4 |
| L4 | 6 | 1.5 |
| L5 | 8 | 1.5 |

Los 18 incidentes del histórico se distribuyen así (H = hurtos, A = accidentes, I = incendios):

| Nodo | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| Incidentes | — | H1 | H1 A2 I1 | H3 | H2 A1 | A1 | A1 I1 | H1 | H1 A1 I1 |

## Reglas del modelo

1. **Costo.** Construir una estación de un tipo en un lote cuesta `costo_base del tipo × factor_zona del lote`, redondeado al millón más cercano. Por ejemplo, un CAI en el lote L1 de la instancia `pequena` cuesta 1.000 × 1.1 = 1.100 millones.
2. **Un lote, una estación.** En cada lote se construye a lo sumo una estación. Sí se pueden construir varias estaciones del mismo tipo en lotes distintos.
3. **Presupuesto.** La suma de los costos de las estaciones construidas no puede superar el presupuesto de la instancia.
4. **Cobertura.** Un incidente queda cubierto si existe una estación **de su mismo tipo** tal que la distancia más corta por la red vial **desde el nodo de la estación hasta el nodo del incidente** es menor o igual al radio de cobertura de ese tipo. La distancia se mide respetando el sentido de los tramos, porque es el vehículo de la estación el que se desplaza hasta el incidente.
5. **Valor de una propuesta.** Es la suma de los pesos de los incidentes cubiertos. Un incidente cubierto por varias estaciones cuenta una sola vez. Por ejemplo, una propuesta que cubre 10 hurtos, 4 accidentes y 1 incendio vale 10 × 1 + 4 × 3 + 1 × 5 = 27.

### Supuestos

- La red vial es una aproximación: se retiraron ciclorrutas, senderos y pasos peatonales identificables por su nombre.
- Los vehículos de emergencia pueden circular por los carriles de Metroplús, que hacen parte de la red.
- Todos los incidentes de un mismo tipo tienen la misma prioridad, sin importar su fecha.

## Restricciones de implementación

- La solución se escribe en Python.
- Se pueden usar `csv`, `json` o `pandas` **solo para leer los archivos**, y `heapq`.
- Para el mapa y la interfaz gráfica se puede usar cualquier librería de visualización o de interfaces (por ejemplo `folium`, `matplotlib`, `streamlit` o `tkinter`), siempre que no se use para calcular rutas, distancias ni combinaciones.
- No se permite usar `networkx`, `osmnx`, `scipy`, `itertools` ni ninguna librería que calcule rutas, distancias en grafos o combinaciones.

## Cronograma y entregables

| Semana | Entrega |
|---|---|
| 1 | Parte 1 |
| 2 | Parte 2 |
| 3 | Parte 3 |
| 4 | Parte 4 y sustentación |

Cada entrega se hace sobre este mismo notebook. Las partes posteriores pueden reutilizar el código de las anteriores.

### Sustentación

En la sustentación se entregará a cada equipo una nueva distribución de lotes y un nuevo histórico de incidentes, con el mismo formato de archivos descrito en la sección "Los datos". El equipo deberá ejecutar su solución sobre esos datos durante la sustentación y ajustarla si es necesario, así que conviene que el código lea cualquier instancia desde sus archivos y no dependa de valores particulares de las instancias entregadas.

### Nota: feria de proyectos de ingeniería

Los equipos que quieran presentar esta práctica en la feria de proyectos de ingeniería deben, además de todo lo anterior, desarrollar una **aplicación interactiva con interfaz gráfica de usuario**, por ejemplo una que permita elegir la instancia, cambiar el presupuesto o los pesos de prioridad y ver la propuesta resultante sobre el mapa. Sin esta aplicación el proyecto no podrá competir.

El equipo ganador de la feria quedará **eximido del examen final**.

## Parte 1. Alcance de cada estación posible

Para una instancia cualquiera, calcula para **cada lote y cada tipo de estación**:

- el costo de construir esa estación en ese lote, y
- la lista de `id` de los incidentes que esa estación cubriría por sí sola, según la regla 4.

### Entrada

El nombre de una instancia (`pequena`, `comuna` o `ciudad`). El programa lee los cinco archivos de su carpeta.

### Salida

Para cada par (lote, tipo): su costo en millones y la lista ordenada de `id` de los incidentes que cubre.

### Restricción adicional

El cálculo para la instancia `ciudad`, con sus 80 lotes y 3 tipos, debe terminar en menos de un minuto.

In [1]:
casos_ej_001_parte1 = [
    {
        "entrada": {"instancia": "pequena", "lote": "L3", "tipo": "accidente"},
        "salida": {"costo": 2800, "incidentes_cubiertos": [1, 2, 4, 6]},
    },
    {
        "entrada": {"instancia": "pequena", "lote": "L2", "tipo": "hurto"},
        "salida": {"costo": 1500, "incidentes_cubiertos": [9, 10, 11, 12, 13]},
    },
]

### Solución de la Parte 1

Se leen los cinco archivos de la instancia, se calcula con Dijkstra (`heapq`) la distancia desde cada lote
hasta los nodos que quedan dentro del radio máximo, y para cada tipo de estación se filtran los incidentes de su
tipo que quedan dentro de su propio radio. Como el grafo es dirigido, la distancia se mide **desde** el nodo de
la estación **hasta** el del incidente.

In [2]:
import copy
import html
import time

from IPython.display import HTML, Markdown, display

In [3]:
import csv
import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

def buscar_carpeta_datos():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for nombre in ("instancias", "datos"):
            if (carpeta / nombre / "pequena").is_dir():
                return carpeta / nombre
    raise FileNotFoundError("No se encontró la carpeta con las instancias (instancias/ o datos/)")


CARPETA_DATOS = buscar_carpeta_datos()


@dataclass
class Instancia:
    nombre: str
    coordenadas: dict
    adyacencia: dict
    lotes: list
    incidentes: list
    parametros: dict

    @property
    def tipos(self):
        return list(self.parametros["tipos"])

    @property
    def presupuesto(self):
        return self.parametros["presupuesto_millones"]


def _leer_csv(ruta):
    with open(ruta, newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))


def cargar_instancia(nombre, carpeta_datos=CARPETA_DATOS):
    carpeta = Path(carpeta_datos) / nombre

    coordenadas = {}
    for fila in _leer_csv(carpeta / "nodos.csv"):
        coordenadas[int(fila["id"])] = (float(fila["lat"]), float(fila["lon"]))

    adyacencia = defaultdict(list)
    for fila in _leer_csv(carpeta / "aristas.csv"):
        adyacencia[int(fila["origen"])].append(
            (int(fila["destino"]), float(fila["longitud_m"]))
        )

    lotes = [
        (fila["lote"], int(fila["nodo"]), float(fila["factor_zona"]))
        for fila in _leer_csv(carpeta / "lotes.csv")
    ]

    incidentes = [
        (int(fila["id"]), int(fila["nodo"]), fila["tipo"])
        for fila in _leer_csv(carpeta / "incidentes.csv")
    ]

    with open(carpeta / "parametros.json", encoding="utf-8") as archivo:
        parametros = json.load(archivo)

    return Instancia(
        nombre=nombre,
        coordenadas=coordenadas,
        adyacencia=adyacencia,
        lotes=lotes,
        incidentes=incidentes,
        parametros=parametros,
    )

In [4]:
import heapq

TOLERANCIA = 1e-9


def distancias_desde(adyacencia, origen, radio_max):
    distancias = {origen: 0.0}
    cola = [(0.0, origen)]

    while cola:
        distancia, nodo = heapq.heappop(cola)
        if distancia > distancias[nodo]:
            continue
        for vecino, longitud in adyacencia.get(nodo, ()):
            nueva = distancia + longitud
            if nueva > radio_max + TOLERANCIA:
                continue
            if nueva < distancias.get(vecino, float("inf")):
                distancias[vecino] = nueva
                heapq.heappush(cola, (nueva, vecino))

    return distancias

In [5]:
from collections import defaultdict



def costo_estacion(costo_base, factor_zona):
    return round(costo_base * factor_zona)


def incidentes_por_tipo_y_nodo(instancia):
    indice = {tipo: defaultdict(list) for tipo in instancia.tipos}
    for id_incidente, nodo, tipo in instancia.incidentes:
        indice[tipo][nodo].append(id_incidente)
    return indice


def calcular_alcances(instancia):
    tipos = instancia.parametros["tipos"]
    radio_max = max(datos["radio_m"] for datos in tipos.values())
    indice = incidentes_por_tipo_y_nodo(instancia)

    alcances = {}
    for codigo, nodo, factor in instancia.lotes:
        distancias = distancias_desde(instancia.adyacencia, nodo, radio_max)
        for tipo, datos in tipos.items():
            limite = datos["radio_m"] + TOLERANCIA
            cubiertos = []
            for nodo_alcanzado, distancia in distancias.items():
                if distancia <= limite:
                    cubiertos.extend(indice[tipo].get(nodo_alcanzado, ()))
            alcances[(codigo, tipo)] = {
                "costo": costo_estacion(datos["costo_base"], factor),
                "incidentes": sorted(cubiertos),
            }
    return alcances

#### Verificación con los casos del enunciado

In [6]:
for caso in casos_ej_001_parte1:
    entrada, esperado = caso["entrada"], caso["salida"]
    instancia = cargar_instancia(entrada["instancia"])
    resultado = calcular_alcances(instancia)[(entrada["lote"], entrada["tipo"])]
    coincide = (
        resultado["costo"] == esperado["costo"]
        and resultado["incidentes"] == esperado["incidentes_cubiertos"]
    )
    print(f"{entrada['lote']} {entrada['tipo']}: costo {resultado['costo']}, "
          f"incidentes {resultado['incidentes']} -> coincide con el enunciado: {coincide}")

L3 accidente: costo 2800, incidentes [1, 2, 4, 6] -> coincide con el enunciado: True
L2 hurto: costo 1500, incidentes [9, 10, 11, 12, 13] -> coincide con el enunciado: True


#### Resultado en la instancia `pequena`

In [7]:
instancia = cargar_instancia("pequena")
alcances = calcular_alcances(instancia)

print(f"{'lote':<6}{'tipo':<11}{'costo':>7}  incidentes que cubre")
for (lote, tipo), datos in alcances.items():
    print(f"{lote:<6}{tipo:<11}{datos['costo']:>7}  {datos['incidentes']}")

lote  tipo         costo  incidentes que cubre
L1    hurto         1100  [7, 8, 12, 13]
L1    accidente     2200  [1, 2, 3, 4]
L1    incendio      5500  [16, 17, 18]
L2    hurto         1500  [9, 10, 11, 12, 13]
L2    accidente     3000  [3, 4, 5]
L2    incendio      7500  [16, 17, 18]
L3    hurto         1400  [8, 15]
L3    accidente     2800  [1, 2, 4, 6]
L3    incendio      7000  [16, 17, 18]
L4    hurto         1500  [9, 10, 11, 14]
L4    accidente     3000  [3, 5, 6]
L4    incendio      7500  [17, 18]
L5    hurto         1500  [14, 15]
L5    accidente     3000  [3, 5, 6]
L5    incendio      7500  [17, 18]


#### Tiempo de cálculo en las instancias grandes (la restricción es menos de un minuto para `ciudad`)

In [8]:
for nombre in ("comuna", "ciudad"):
    instancia = cargar_instancia(nombre)
    inicio = time.perf_counter()
    alcances = calcular_alcances(instancia)
    segundos = time.perf_counter() - inicio
    print(f"{nombre}: {len(instancia.lotes)} lotes x {len(instancia.tipos)} tipos = "
          f"{len(alcances)} combinaciones en {segundos:.2f} s")

comuna: 15 lotes x 3 tipos = 45 combinaciones en 0.06 s


ciudad: 80 lotes x 3 tipos = 240 combinaciones en 0.51 s


## Parte 2. Propuestas construidas paso a paso

### 2.1 Heurística base

La Secretaría de Planeación usa hoy el siguiente procedimiento para armar una propuesta:

1. Empezar sin estaciones y con todo el presupuesto disponible.
2. Considerar todas las estaciones posibles (lote, tipo) que cumplen dos condiciones: el lote todavía está libre y su costo no supera el presupuesto restante.
3. De ellas, elegir la que **agrega más valor**, es decir, la que tiene mayor suma de pesos de los incidentes que cubre y que **todavía no están cubiertos** por las estaciones ya elegidas. Si hay empate, se elige la de menor código de lote; si el empate es dentro del mismo lote, se prefiere en el orden hurto, accidente, incendio.
4. Si ninguna estación posible agrega valor, terminar. Si no, construirla, descontar su costo, marcar el lote como ocupado y volver al paso 2.

Implementa la heurística base y aplícala a las tres instancias.

### 2.2 Heurísticas propias

Diseña al menos una heurística propia que también construya la propuesta eligiendo una estación a la vez, y que obtenga un valor mayor que la base en al menos una de las tres instancias. Aplícala a las tres instancias.

Para cada heurística, la base incluida, construye a mano una instancia pequeña (en el mismo formato de archivos, con no más de 6 nodos) en la que la heurística **no** obtenga la mejor propuesta posible. Explica qué decisión de la heurística la lleva por mal camino en ese caso.

### Salida

Para cada heurística e instancia: las estaciones elegidas (lote y tipo, ordenadas por código de lote), el costo total, la cantidad de casos cubiertos de cada tipo y el valor de la propuesta.

In [9]:
casos_ej_001_parte2 = [
    {
        "entrada": {"instancia": "pequena", "heuristica": "base"},
        "salida": {
            "estaciones": [("L1", "incendio")],
            "costo_total": 5500,
            "casos_cubiertos": {"hurto": 0, "accidente": 0, "incendio": 3},
            "valor": 15,
        },
    },
    {
        "entrada": {"instancia": "comuna", "heuristica": "base"},
        "salida": {
            "estaciones": [
                ("L05", "accidente"), ("L06", "hurto"), ("L08", "accidente"),
                ("L09", "incendio"), ("L13", "hurto"), ("L14", "hurto"),
                ("L15", "accidente"),
            ],
            "costo_total": 11950,
            "casos_cubiertos": {"hurto": 486, "accidente": 296, "incendio": 31},
            "valor": 1529,
        },
    },
]

### Solución de la Parte 2

Las dos heurísticas comparten el mismo ciclo: en cada vuelta consideran las estaciones cuyo lote sigue libre y
cuyo costo cabe en el presupuesto restante, calculan el valor que cada una **agrega** (solo incidentes aún no
cubiertos), construyen la mejor según un criterio y repiten hasta que ninguna agregue valor. Lo único que cambia
es el criterio:

- **Base:** mayor valor agregado. Los empates se resuelven por menor código de lote y luego por el orden
  hurto, accidente, incendio.
- **Eficiencia (propia):** mayor valor agregado por millón de costo.

In [10]:
def criterio_base(valor_agregado, costo):
    return valor_agregado


def criterio_eficiencia(valor_agregado, costo):
    return valor_agregado / costo


HEURISTICAS = {
    "base": criterio_base,
    "eficiencia": criterio_eficiencia,
}


def construir_propuesta(instancia, alcances, criterio):
    pesos = {tipo: datos["peso"] for tipo, datos in instancia.parametros["tipos"].items()}
    orden_tipos = {tipo: posicion for posicion, tipo in enumerate(instancia.tipos)}
    candidatas = sorted(alcances, key=lambda par: (par[0], orden_tipos[par[1]]))

    lotes_libres = {codigo for codigo, _, _ in instancia.lotes}
    presupuesto_restante = instancia.presupuesto
    cubiertos = set()
    elegidas = []

    while True:
        mejor = None
        mejor_clave = None

        for lote, tipo in candidatas:
            if lote not in lotes_libres:
                continue
            costo = alcances[(lote, tipo)]["costo"]
            if costo > presupuesto_restante:
                continue
            nuevos = [i for i in alcances[(lote, tipo)]["incidentes"] if i not in cubiertos]
            valor_agregado = len(nuevos) * pesos[tipo]
            if valor_agregado == 0:
                continue
            clave = criterio(valor_agregado, costo)
            if mejor_clave is None or clave > mejor_clave:
                mejor, mejor_clave = (lote, tipo), clave

        if mejor is None:
            break

        lote, tipo = mejor
        elegidas.append(mejor)
        lotes_libres.discard(lote)
        presupuesto_restante -= alcances[mejor]["costo"]
        cubiertos.update(alcances[mejor]["incidentes"])

    return sorted(elegidas)


def resumir_propuesta(instancia, alcances, estaciones):
    pesos = {tipo: datos["peso"] for tipo, datos in instancia.parametros["tipos"].items()}
    cubiertos = set()
    casos = {tipo: 0 for tipo in instancia.tipos}
    costo_total = 0

    for lote, tipo in sorted(estaciones):
        costo_total += alcances[(lote, tipo)]["costo"]
        for incidente in alcances[(lote, tipo)]["incidentes"]:
            if incidente not in cubiertos:
                cubiertos.add(incidente)
                casos[tipo] += 1

    valor = sum(casos[tipo] * pesos[tipo] for tipo in casos)
    return {
        "estaciones": sorted(estaciones),
        "costo_total": costo_total,
        "casos_cubiertos": casos,
        "valor": valor,
    }

In [11]:
resultados_heuristicas = {}

for nombre in ("pequena", "comuna", "ciudad"):
    instancia = cargar_instancia(nombre)
    alcances = calcular_alcances(instancia)
    print(f"=== {nombre} (presupuesto {instancia.presupuesto}) ===")
    for etiqueta, criterio in HEURISTICAS.items():
        inicio = time.perf_counter()
        estaciones = construir_propuesta(instancia, alcances, criterio)
        segundos = time.perf_counter() - inicio
        resumen = resumir_propuesta(instancia, alcances, estaciones)
        resultados_heuristicas[(nombre, etiqueta)] = (resumen, segundos)

        print(f"\nHeurística {etiqueta}")
        print("  estaciones:", ", ".join(f"{lote}-{tipo}" for lote, tipo in resumen["estaciones"]))
        print("  costo total:", resumen["costo_total"])
        print("  casos cubiertos:", resumen["casos_cubiertos"])
        print("  valor:", resumen["valor"])
    print()

=== pequena (presupuesto 6000) ===

Heurística base
  estaciones: L1-incendio
  costo total: 5500
  casos cubiertos: {'hurto': 0, 'accidente': 0, 'incendio': 3}
  valor: 15

Heurística eficiencia
  estaciones: L1-accidente, L2-hurto, L3-hurto
  costo total: 5100
  casos cubiertos: {'hurto': 7, 'accidente': 4, 'incendio': 0}
  valor: 19

=== comuna (presupuesto 12000) ===

Heurística base
  estaciones: L05-accidente, L06-hurto, L08-accidente, L09-incendio, L13-hurto, L14-hurto, L15-accidente
  costo total: 11950
  casos cubiertos: {'hurto': 486, 'accidente': 296, 'incendio': 31}
  valor: 1529

Heurística eficiencia
  estaciones: L02-hurto, L03-accidente, L05-accidente, L06-hurto, L09-hurto, L10-accidente, L13-hurto, L14-hurto, L15-accidente
  costo total: 11600
  casos cubiertos: {'hurto': 532, 'accidente': 320, 'incendio': 0}
  valor: 1492



=== ciudad (presupuesto 40000) ===

Heurística base
  estaciones: L01-accidente, L04-hurto, L05-accidente, L08-incendio, L11-hurto, L12-accidente, L14-accidente, L19-accidente, L28-incendio, L29-incendio, L32-accidente, L34-accidente, L35-hurto, L37-accidente, L46-hurto, L49-accidente, L57-accidente, L62-hurto, L67-accidente, L70-hurto
  costo total: 39850
  casos cubiertos: {'hurto': 1350, 'accidente': 1757, 'incendio': 194}
  valor: 7591

Heurística eficiencia
  estaciones: L01-accidente, L04-hurto, L08-hurto, L11-hurto, L14-hurto, L19-accidente, L25-accidente, L26-accidente, L28-incendio, L29-incendio, L32-accidente, L34-hurto, L35-hurto, L37-accidente, L39-accidente, L45-accidente, L46-hurto, L48-hurto, L57-accidente, L62-hurto, L67-accidente, L69-accidente, L70-hurto
  costo total: 39950
  casos cubiertos: {'hurto': 1887, 'accidente': 1718, 'incendio': 133}
  valor: 7706



#### Verificación de la heurística base con los casos del enunciado

In [12]:
for caso in casos_ej_001_parte2:
    entrada, esperado = caso["entrada"], caso["salida"]
    obtenido = resultados_heuristicas[(entrada["instancia"], entrada["heuristica"])][0]
    print(f"{entrada['instancia']} / {entrada['heuristica']}: valor {obtenido['valor']} "
          f"-> coincide con el enunciado: {obtenido == esperado}")

pequena / base: valor 15 -> coincide con el enunciado: True
comuna / base: valor 1529 -> coincide con el enunciado: True


#### Instancias manuales donde cada heurística no obtiene la mejor propuesta

Se escriben en el mismo formato de archivos del enunciado, en la carpeta `instancias_manuales`.

**Contraejemplo de la base** (4 nodos en línea, tramos de 2100 m, presupuesto 6.000). Con esos tramos, un CAI o
una estación de tránsito solo cubre el nodo donde está, y solo el lote L3 alcanza el nodo 3 con bomberos. La base
elige primero bomberos en L3: agrega 15 puntos (3 incendios), frente a 6 de cualquier estación de tránsito. Esa
estación cuesta 5.000 y los 1.000 restantes no alcanzan para nada que cubra algo, así que termina con 15. La mejor
propuesta son tres estaciones de tránsito (L1, L2 y L3), que cuestan 6.000 y cubren 6 accidentes: 18 puntos. La
decisión que la desvía es mirar solo el valor agregado y no el costo: gasta casi todo el presupuesto en una sola
estación.

**Contraejemplo de la eficiencia** (2 nodos, tramo de 2100 m, presupuesto 2.000). La eficiencia elige primero un
CAI en L1: agrega 4 y cuesta 1.000 (0,004 por millón), frente a 0,003 de tránsito en L2 (agrega 6 y cuesta 2.000).
Quedan 1.000, que no alcanzan para tránsito ni cubren nada más: termina con 4. La mejor propuesta es tránsito en
L2: 6 puntos. La decisión que la desvía es preferir la mejor relación valor por millón aunque sea una compra
pequeña: deja un presupuesto sobrante inútil para la estación grande.

In [13]:
CARPETA_MANUALES = CARPETA_DATOS.parent / "instancias_manuales"


def escribir_csv(ruta, encabezado, filas):
    with open(ruta, "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo, lineterminator="\n")
        escritor.writerow(encabezado)
        escritor.writerows(filas)


def escribir_instancia(nombre, nodos, aristas, lotes, incidentes, presupuesto, descripcion):
    carpeta = CARPETA_MANUALES / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    escribir_csv(carpeta / "nodos.csv", ["id", "lat", "lon"], nodos)
    escribir_csv(carpeta / "aristas.csv", ["origen", "destino", "longitud_m", "nombre"], aristas)
    escribir_csv(carpeta / "lotes.csv", ["lote", "nodo", "factor_zona"], lotes)
    escribir_csv(carpeta / "incidentes.csv", ["id", "nodo", "tipo", "fecha_hora"], incidentes)

    parametros = copy.deepcopy(cargar_instancia("pequena").parametros)
    parametros["descripcion"] = descripcion
    parametros["presupuesto_millones"] = presupuesto
    with open(carpeta / "parametros.json", "w", encoding="utf-8") as archivo:
        json.dump(parametros, archivo, ensure_ascii=False, indent=2)


escribir_instancia(
    "contraejemplo_base",
    nodos=[(0, 6.25, -75.590), (1, 6.25, -75.571), (2, 6.25, -75.552), (3, 6.25, -75.533)],
    aristas=[(0, 1, 2100, "Calle 1"), (1, 0, 2100, "Calle 1"), (1, 2, 2100, "Calle 1"),
             (2, 1, 2100, "Calle 1"), (2, 3, 2100, "Calle 1"), (3, 2, 2100, "Calle 1")],
    lotes=[("L1", 0, 1.0), ("L2", 1, 1.0), ("L3", 2, 1.0)],
    incidentes=[(1, 0, "accidente", "2025-01-01 08:00"), (2, 0, "accidente", "2025-01-02 08:00"),
                (3, 1, "accidente", "2025-01-03 08:00"), (4, 1, "accidente", "2025-01-04 08:00"),
                (5, 2, "accidente", "2025-01-05 08:00"), (6, 2, "accidente", "2025-01-06 08:00"),
                (7, 3, "incendio", "2025-01-07 08:00"), (8, 3, "incendio", "2025-01-08 08:00"),
                (9, 3, "incendio", "2025-01-09 08:00")],
    presupuesto=6000,
    descripcion="Contraejemplo de la heuristica base: 4 nodos en linea, tramos de 2100 m.",
)

escribir_instancia(
    "contraejemplo_eficiencia",
    nodos=[(0, 6.25, -75.590), (1, 6.25, -75.571)],
    aristas=[(0, 1, 2100, "Calle 1"), (1, 0, 2100, "Calle 1")],
    lotes=[("L1", 0, 1.0), ("L2", 1, 1.0)],
    incidentes=[(1, 0, "hurto", "2025-01-01 08:00"), (2, 0, "hurto", "2025-01-02 08:00"),
                (3, 0, "hurto", "2025-01-03 08:00"), (4, 0, "hurto", "2025-01-04 08:00"),
                (5, 1, "accidente", "2025-01-05 08:00"), (6, 1, "accidente", "2025-01-06 08:00")],
    presupuesto=2000,
    descripcion="Contraejemplo de la heuristica de eficiencia: 2 nodos, tramo de 2100 m.",
)

print("Instancias manuales escritas en", CARPETA_MANUALES)

Instancias manuales escritas en /home/claude/alcaldia-estaciones/instancias_manuales


In [14]:
casos_contraejemplo = [
    ("contraejemplo_base", "base", [("L1", "accidente"), ("L2", "accidente"), ("L3", "accidente")]),
    ("contraejemplo_eficiencia", "eficiencia", [("L2", "accidente")]),
]

for nombre, etiqueta, mejor in casos_contraejemplo:
    instancia = cargar_instancia(nombre, carpeta_datos=CARPETA_MANUALES)
    alcances = calcular_alcances(instancia)
    elegida = resumir_propuesta(
        instancia, alcances, construir_propuesta(instancia, alcances, HEURISTICAS[etiqueta])
    )
    alternativa = resumir_propuesta(instancia, alcances, mejor)
    print(f"{nombre} (presupuesto {instancia.presupuesto})")
    print(f"  heurística {etiqueta}: {elegida['estaciones']} -> costo {elegida['costo_total']}, "
          f"valor {elegida['valor']}")
    print(f"  propuesta mejor:      {alternativa['estaciones']} -> costo {alternativa['costo_total']}, "
          f"valor {alternativa['valor']}")
    print()

contraejemplo_base (presupuesto 6000)
  heurística base: [('L3', 'incendio')] -> costo 5000, valor 15
  propuesta mejor:      [('L1', 'accidente'), ('L2', 'accidente'), ('L3', 'accidente')] -> costo 6000, valor 18

contraejemplo_eficiencia (presupuesto 2000)
  heurística eficiencia: [('L1', 'hurto')] -> costo 1000, valor 4
  propuesta mejor:      [('L2', 'accidente')] -> costo 2000, valor 6



## Parte 3. La mejor propuesta posible

Obtén la propuesta de **mayor valor posible** que respete todas las reglas, para las instancias `pequena` y `comuna`. Tu programa debe garantizar que no existe una propuesta mejor, no solo encontrar una buena.

Luego compara en una tabla, para las tres instancias, el valor y el tiempo de ejecución de la heurística base, de tus heurísticas propias y de la mejor propuesta posible. En la instancia `ciudad`, explica con un cálculo por qué no reportas la mejor propuesta posible.

### Salida

Para `pequena` y `comuna`: las estaciones de la mejor propuesta (ordenadas por código de lote), su costo total, los casos cubiertos de cada tipo, su valor y el tiempo que tardó en obtenerse. Además, la tabla comparativa.

In [15]:
casos_ej_001_parte3 = [
    {
        "entrada": {"instancia": "pequena"},
        "salida": {
            "estaciones": [("L1", "hurto"), ("L3", "accidente"), ("L4", "hurto")],
            "costo_total": 5400,
            "casos_cubiertos": {"hurto": 8, "accidente": 4, "incendio": 0},
            "valor": 20,
        },
    },
]

### Solución de la Parte 3

Búsqueda exacta por **ramificación y poda**. Se recorren los lotes uno por uno y en cada uno se prueba cada
estación posible o ninguna. Antes de seguir por una rama se calcula una cota superior de lo que aún podría
ganarse con el presupuesto restante, tomando las estaciones por valor agregado por millón y permitiendo comprar
una fracción de la última (mochila fraccionaria). Esa cota nunca es menor que el valor real alcanzable, así que
descartar una rama cuando no puede superar la mejor propuesta conocida **nunca descarta el óptimo**: de ahí sale
la garantía. La búsqueda arranca con el mejor valor de las heurísticas para podar desde el principio, y los
incidentes cubiertos por cada estación se guardan como máscaras de bits.

In [16]:
PROPUESTAS_POR_SEGUNDO = 1e9
SEGUNDOS_POR_ANIO = 365.25 * 24 * 3600


def espacio_de_busqueda(instancia):
    opciones_por_lote = len(instancia.tipos) + 1
    combinaciones = opciones_por_lote ** len(instancia.lotes)
    segundos = combinaciones / PROPUESTAS_POR_SEGUNDO
    return {
        "lotes": len(instancia.lotes),
        "opciones_por_lote": opciones_por_lote,
        "combinaciones": combinaciones,
        "anios": segundos / SEGUNDOS_POR_ANIO,
    }


def _opciones_por_lote(instancia, alcances):
    pesos = {tipo: datos["peso"] for tipo, datos in instancia.parametros["tipos"].items()}

    posicion = {tipo: {} for tipo in instancia.tipos}
    for id_incidente, _, tipo in sorted(instancia.incidentes):
        posicion[tipo][id_incidente] = len(posicion[tipo])

    por_lote = {}
    for (lote, tipo), datos in alcances.items():
        if datos["costo"] > instancia.presupuesto or not datos["incidentes"]:
            continue
        mascara = 0
        for incidente in datos["incidentes"]:
            mascara |= 1 << posicion[tipo][incidente]
        por_lote.setdefault(lote, []).append(
            {"lote": lote, "tipo": tipo, "costo": datos["costo"],
             "mascara": mascara, "peso": pesos[tipo]}
        )

    lotes = sorted(por_lote)
    return [por_lote[lote] for lote in lotes]


def _ganancia(opcion, cubiertos):
    nuevos = opcion["mascara"] & ~cubiertos[opcion["tipo"]]
    return nuevos.bit_count() * opcion["peso"]


def _cota_superior(grupos, desde, presupuesto, cubiertos):
    items = []
    for grupo in grupos[desde:]:
        for opcion in grupo:
            if opcion["costo"] > presupuesto:
                continue
            ganancia = _ganancia(opcion, cubiertos)
            if ganancia > 0:
                items.append((ganancia / opcion["costo"], ganancia, opcion["costo"]))

    items.sort(reverse=True)
    total = 0.0
    restante = presupuesto
    for _, ganancia, costo in items:
        if costo <= restante:
            total += ganancia
            restante -= costo
        else:
            total += ganancia * restante / costo
            break
    return total


def mejor_propuesta(instancia, alcances):
    grupos = _opciones_por_lote(instancia, alcances)

    mejor = {"valor": -1, "estaciones": []}
    for criterio in HEURISTICAS.values():
        estaciones = construir_propuesta(instancia, alcances, criterio)
        valor = resumir_propuesta(instancia, alcances, estaciones)["valor"]
        if valor > mejor["valor"]:
            mejor = {"valor": valor, "estaciones": estaciones}

    nodos = [0]

    def buscar(indice, presupuesto, cubiertos, valor, elegidas):
        nodos[0] += 1
        if valor > mejor["valor"]:
            mejor["valor"] = valor
            mejor["estaciones"] = sorted((o["lote"], o["tipo"]) for o in elegidas)

        if indice == len(grupos):
            return
        if valor + _cota_superior(grupos, indice, presupuesto, cubiertos) <= mejor["valor"]:
            return

        opciones = [o for o in grupos[indice] if o["costo"] <= presupuesto]
        opciones.sort(key=lambda o: _ganancia(o, cubiertos), reverse=True)
        for opcion in opciones:
            ganancia = _ganancia(opcion, cubiertos)
            if ganancia == 0:
                continue
            nuevos_cubiertos = dict(cubiertos)
            nuevos_cubiertos[opcion["tipo"]] |= opcion["mascara"]
            elegidas.append(opcion)
            buscar(indice + 1, presupuesto - opcion["costo"], nuevos_cubiertos,
                   valor + ganancia, elegidas)
            elegidas.pop()

        buscar(indice + 1, presupuesto, cubiertos, valor, elegidas)

    buscar(0, instancia.presupuesto, {tipo: 0 for tipo in instancia.tipos}, 0, [])
    return mejor["estaciones"], nodos[0]

In [17]:
optimos = {}

for nombre in ("pequena", "comuna"):
    instancia = cargar_instancia(nombre)
    alcances = calcular_alcances(instancia)
    inicio = time.perf_counter()
    estaciones, nodos = mejor_propuesta(instancia, alcances)
    segundos = time.perf_counter() - inicio
    resumen = resumir_propuesta(instancia, alcances, estaciones)
    optimos[nombre] = (resumen, segundos)

    print(f"=== {nombre} (presupuesto {instancia.presupuesto}) ===")
    print(f"Mejor propuesta posible, obtenida en {segundos:.2f} s ({nodos} nodos explorados)")
    print("  estaciones:", ", ".join(f"{lote}-{tipo}" for lote, tipo in resumen["estaciones"]))
    print("  costo total:", resumen["costo_total"])
    print("  casos cubiertos:", resumen["casos_cubiertos"])
    print("  valor:", resumen["valor"])
    print()

esperado = casos_ej_001_parte3[0]["salida"]
print("pequena coincide con el enunciado:", optimos["pequena"][0] == esperado)

=== pequena (presupuesto 6000) ===
Mejor propuesta posible, obtenida en 0.00 s (41 nodos explorados)
  estaciones: L1-hurto, L3-accidente, L4-hurto
  costo total: 5400
  casos cubiertos: {'hurto': 8, 'accidente': 4, 'incendio': 0}
  valor: 20



=== comuna (presupuesto 12000) ===
Mejor propuesta posible, obtenida en 3.47 s (649726 nodos explorados)
  estaciones: L03-accidente, L06-hurto, L09-hurto, L11-hurto, L13-accidente, L14-hurto, L15-incendio
  costo total: 11700
  casos cubiertos: {'hurto': 519, 'accidente': 300, 'incendio': 31}
  valor: 1574

pequena coincide con el enunciado: True


#### Por qué no se reporta la mejor propuesta posible en `ciudad`

Cada lote admite cuatro opciones (vacío, CAI, tránsito o bomberos), así que el número de combinaciones es
4 elevado al número de lotes. La poda reduce mucho el trabajo (en `comuna` explora cerca de 650 mil nodos en
lugar de 4¹⁵, unos mil millones), pero con 80 lotes el espacio es de 4⁸⁰ y no alcanza.

In [18]:
espacio = espacio_de_busqueda(cargar_instancia("ciudad"))
print(f"Lotes: {espacio['lotes']}, opciones por lote: {espacio['opciones_por_lote']} "
      "(vacío, CAI, tránsito o bomberos)")
print(f"Combinaciones: {espacio['opciones_por_lote']}^{espacio['lotes']} = {espacio['combinaciones']:.2e}")
print(f"A mil millones de propuestas por segundo: {espacio['anios']:.2e} años")

Lotes: 80, opciones por lote: 4 (vacío, CAI, tránsito o bomberos)
Combinaciones: 4^80 = 1.46e+48
A mil millones de propuestas por segundo: 4.63e+31 años


#### Tabla comparativa (los tiempos no incluyen el cálculo de alcances de la Parte 1)

In [19]:
def tabla_comparativa():
    lineas = [
        "| Instancia | Base: valor | Base: tiempo (s) | Eficiencia: valor | Eficiencia: tiempo (s) "
        "| Óptimo: valor | Óptimo: tiempo (s) |",
        "|---|---|---|---|---|---|---|",
    ]
    for nombre in ("pequena", "comuna", "ciudad"):
        base, t_base = resultados_heuristicas[(nombre, "base")]
        eficiencia, t_eficiencia = resultados_heuristicas[(nombre, "eficiencia")]
        if nombre in optimos:
            optimo_resumen, t_optimo = optimos[nombre]
            columnas_optimo = f"{optimo_resumen['valor']} | {t_optimo:.2f}"
        else:
            columnas_optimo = "no calculado | -"
        lineas.append(
            f"| {nombre} | {base['valor']} | {t_base:.4f} | {eficiencia['valor']} "
            f"| {t_eficiencia:.4f} | {columnas_optimo} |"
        )
    return "\n".join(lineas)


display(Markdown(tabla_comparativa()))

| Instancia | Base: valor | Base: tiempo (s) | Eficiencia: valor | Eficiencia: tiempo (s) | Óptimo: valor | Óptimo: tiempo (s) |
|---|---|---|---|---|---|---|
| pequena | 15 | 0.0000 | 19 | 0.0000 | 20 | 0.00 |
| comuna | 1529 | 0.0010 | 1492 | 0.0011 | 1574 | 3.47 |
| ciudad | 7591 | 0.0124 | 7706 | 0.0128 | no calculado | - |

## Parte 4. Propuesta para el concejo

Elige la propuesta que recomendarías para la instancia `ciudad`, obtenida con cualquiera de tus heurísticas, y preséntala con:

1. **Un mapa** de la ciudad, hecho con `folium` o con cualquier otra librería, donde se distingan los lotes no usados, las estaciones construidas según su tipo, y los incidentes cubiertos y no cubiertos. Mostrar el resultado de forma gráfica sobre un mapa es obligatorio para todos los equipos, participen o no en la feria.
2. **Una tabla** con cada estación construida: lote, tipo, costo y casos que cubre. Al final, el costo total frente al presupuesto, los casos cubiertos de cada tipo y el valor de la propuesta.
3. **Un párrafo** que justifique por qué recomiendas esa propuesta y no otra de las que obtuviste.

Repite el mapa y la tabla para la mejor propuesta posible de la instancia `comuna`.

### Solución de la Parte 4

Para `ciudad` se recomienda la propuesta de mayor valor entre las heurísticas. Para `comuna` se presenta la mejor
propuesta posible de la Parte 3. En el mapa se distinguen los lotes no usados, las estaciones por tipo y los
incidentes cubiertos y no cubiertos; las capas se pueden activar y desactivar desde el panel de la esquina
superior derecha. El mapa también queda guardado como archivo HTML en la carpeta `resultados`.

In [20]:
from pathlib import Path

import folium

COLOR_ESTACION = {"hurto": "blue", "accidente": "orange", "incendio": "red"}
COLOR_LEYENDA = {"hurto": "#38a9dc", "accidente": "#f69730", "incendio": "#d43e2a"}
ICONO_ESTACION = {"hurto": "shield", "accidente": "car", "incendio": "fire"}

FONDO_URL = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
)

COLOR_CUBIERTO = "#1a9850"
COLOR_NO_CUBIERTO = "#333333"


def incidentes_cubiertos(alcances, estaciones):
    cubiertos = set()
    for estacion in estaciones:
        cubiertos.update(alcances[estacion]["incidentes"])
    return cubiertos


def _leyenda(titulo, instancia, estaciones_construidas, cubiertos):
    tipos = instancia.parametros["tipos"]
    filas = []
    for tipo, datos in tipos.items():
        color = COLOR_LEYENDA.get(tipo, "#888888")
        filas.append(
            f'<div><span style="background:{color};display:inline-block;width:12px;'
            f'height:12px;border-radius:50%;margin-right:6px"></span>{datos["estacion"]}</div>'
        )
    filas.append(
        f'<div><span style="background:{COLOR_CUBIERTO};display:inline-block;width:10px;'
        f'height:10px;border-radius:50%;margin-right:8px"></span>Incidente cubierto</div>'
    )
    filas.append(
        f'<div><span style="background:{COLOR_NO_CUBIERTO};display:inline-block;width:10px;'
        f'height:10px;border-radius:50%;margin-right:8px"></span>Incidente no cubierto</div>'
    )
    filas.append(
        '<div><span style="background:#fff;border:2px solid #777;display:inline-block;'
        'width:10px;height:10px;border-radius:50%;margin-right:8px"></span>Lote no usado</div>'
    )
    total = len(instancia.incidentes)
    resumen = (
        f"{len(estaciones_construidas)} estaciones &middot; "
        f"{len(cubiertos)} de {total} incidentes cubiertos"
    )
    html = (
        '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:#fff;'
        'color:#222;padding:10px 14px;border:1px solid #999;border-radius:6px;'
        'font:13px/1.5 sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)">'
        f'<b>{titulo}</b><div style="margin-bottom:6px">{resumen}</div>'
        + "".join(filas)
        + "</div>"
    )
    return folium.Element(html)


def crear_mapa(instancia, alcances, estaciones, ruta_salida, titulo):
    cubiertos = incidentes_cubiertos(alcances, estaciones)
    nodo_de_lote = {codigo: nodo for codigo, nodo, _ in instancia.lotes}
    lotes_usados = {lote for lote, _ in estaciones}

    latitudes = [lat for lat, _ in instancia.coordenadas.values()]
    longitudes = [lon for _, lon in instancia.coordenadas.values()]
    mapa = folium.Map(
        location=[sum(latitudes) / len(latitudes), sum(longitudes) / len(longitudes)],
        tiles=None,
    )
    folium.TileLayer(
        tiles=FONDO_URL,
        attr="Tiles &copy; Esri",
        name="Mapa de calles",
        max_zoom=19,
    ).add_to(mapa)
    mapa.fit_bounds([[min(latitudes), min(longitudes)], [max(latitudes), max(longitudes)]])

    grupo_no_cubiertos = folium.FeatureGroup(name="Incidentes no cubiertos")
    grupo_cubiertos = folium.FeatureGroup(name="Incidentes cubiertos")
    grupo_lotes = folium.FeatureGroup(name="Lotes no usados")
    grupo_estaciones = folium.FeatureGroup(name="Estaciones construidas")

    for id_incidente, nodo, tipo in instancia.incidentes:
        cubierto = id_incidente in cubiertos
        folium.CircleMarker(
            location=instancia.coordenadas[nodo],
            radius=3,
            color=COLOR_CUBIERTO if cubierto else COLOR_NO_CUBIERTO,
            weight=1,
            fill=True,
            fill_color=COLOR_CUBIERTO if cubierto else COLOR_NO_CUBIERTO,
            fill_opacity=0.7,
            tooltip=f"Incidente {id_incidente} ({tipo}): {'cubierto' if cubierto else 'no cubierto'}",
        ).add_to(grupo_cubiertos if cubierto else grupo_no_cubiertos)

    for codigo, nodo, factor in instancia.lotes:
        if codigo in lotes_usados:
            continue
        folium.CircleMarker(
            location=instancia.coordenadas[nodo],
            radius=6,
            color="#777777",
            weight=2,
            fill=True,
            fill_color="#ffffff",
            fill_opacity=1,
            tooltip=f"Lote {codigo} (no usado, factor de zona {factor})",
        ).add_to(grupo_lotes)

    for lote, tipo in sorted(estaciones):
        datos = alcances[(lote, tipo)]
        nombre = instancia.parametros["tipos"][tipo]["estacion"]
        folium.Marker(
            location=instancia.coordenadas[nodo_de_lote[lote]],
            icon=folium.Icon(
                color=COLOR_ESTACION.get(tipo, "gray"),
                icon=ICONO_ESTACION.get(tipo, "info-sign"),
                prefix="fa",
            ),
            tooltip=f"{lote}: {nombre}",
            popup=folium.Popup(
                f"<b>{lote}</b><br>{nombre}<br>Costo: {datos['costo']} millones"
                f"<br>Cubre {len(datos['incidentes'])} incidentes",
                max_width=260,
            ),
        ).add_to(grupo_estaciones)

    for grupo in (grupo_no_cubiertos, grupo_cubiertos, grupo_lotes, grupo_estaciones):
        grupo.add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)
    mapa.get_root().html.add_child(_leyenda(titulo, instancia, estaciones, cubiertos))

    ruta = Path(ruta_salida)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    mapa.save(str(ruta))
    return ruta

In [21]:
def tabla_estaciones(instancia, alcances, resumen):
    lineas = [
        "| Lote | Tipo | Estación | Costo (millones) | Casos que cubre |",
        "|---|---|---|---|---|",
    ]
    for lote, tipo in resumen["estaciones"]:
        datos = alcances[(lote, tipo)]
        nombre = instancia.parametros["tipos"][tipo]["estacion"]
        lineas.append(
            f"| {lote} | {tipo} | {nombre} | {datos['costo']} | {len(datos['incidentes'])} |"
        )

    casos = resumen["casos_cubiertos"]
    lineas.append("")
    lineas.append(
        f"- Costo total: {resumen['costo_total']} millones, frente a un presupuesto de "
        f"{instancia.presupuesto} millones."
    )
    lineas.append(
        f"- Casos cubiertos: {casos['hurto']} hurtos, {casos['accidente']} accidentes "
        f"y {casos['incendio']} incendios."
    )
    lineas.append(f"- Valor de la propuesta: {resumen['valor']}.")
    return "\n".join(lineas)


def _diferencia_de_casos(elegida, otra):
    return ", ".join(
        f"{tipo} {elegida['casos_cubiertos'][tipo] - otra['casos_cubiertos'][tipo]:+d}"
        for tipo in elegida["casos_cubiertos"]
    )


NOMBRES_PROPUESTA = {
    "base": "la propuesta de la heurística base",
    "eficiencia": "la propuesta de la heurística de eficiencia",
    "optima": "la mejor propuesta posible",
}


def _nombre(etiqueta):
    return NOMBRES_PROPUESTA.get(etiqueta, f"la propuesta «{etiqueta}»")


def parrafo_justificacion(instancia, resumenes, elegida, es_optima=False):
    resumen = resumenes[elegida]
    casos = resumen["casos_cubiertos"]
    uso = 100 * resumen["costo_total"] / instancia.presupuesto

    texto = (
        f"Para la instancia {instancia.nombre} se recomienda {_nombre(elegida)}: "
        f"alcanza un valor de {resumen['valor']}, el más alto entre las alternativas obtenidas, "
        f"con un costo de {resumen['costo_total']} millones sobre un presupuesto de "
        f"{instancia.presupuesto} ({uso:.1f} % utilizado), y cubre {casos['hurto']} hurtos, "
        f"{casos['accidente']} accidentes y {casos['incendio']} incendios."
    )

    for etiqueta, otra in resumenes.items():
        if etiqueta == elegida:
            continue
        texto += (
            f" Frente a {_nombre(etiqueta)} (valor {otra['valor']}) suma "
            f"{resumen['valor'] - otra['valor']} puntos, con esta diferencia en casos cubiertos: "
            f"{_diferencia_de_casos(resumen, otra)}."
        )

    if es_optima:
        texto += (
            " Además, la búsqueda exacta garantiza que ninguna propuesta que respete "
            "el presupuesto alcanza un valor mayor."
        )
    else:
        texto += (
            " No se garantiza que sea la mejor propuesta posible: en esta instancia el espacio "
            "de búsqueda es demasiado grande para recorrerlo por completo."
        )
    return texto

In [22]:
CARPETA_RESULTADOS = CARPETA_DATOS.parent / "resultados"


def mostrar_mapa(ruta, alto=620):
    contenido = Path(ruta).read_text(encoding="utf-8")
    marco = html.escape(contenido, quote=True)
    return HTML(f'<iframe srcdoc="{marco}" width="100%" height="{alto}" style="border:none"></iframe>')


def presentar_propuesta(instancia, alcances, resumenes, elegida, es_optima, nombre_archivo, titulo):
    resumen = resumenes[elegida]
    ruta = crear_mapa(
        instancia, alcances, resumen["estaciones"],
        CARPETA_RESULTADOS / f"mapa_{nombre_archivo}.html", titulo,
    )
    display(Markdown(f"#### {titulo}"))
    display(mostrar_mapa(ruta))
    display(Markdown(tabla_estaciones(instancia, alcances, resumen)))
    display(Markdown("**Justificación.** " + parrafo_justificacion(instancia, resumenes, elegida, es_optima)))

In [23]:
instancia = cargar_instancia("ciudad")
alcances = calcular_alcances(instancia)
resumenes = {
    etiqueta: resultados_heuristicas[("ciudad", etiqueta)][0] for etiqueta in HEURISTICAS
}
elegida = max(resumenes, key=lambda etiqueta: resumenes[etiqueta]["valor"])

presentar_propuesta(
    instancia, alcances, resumenes, elegida, False, "ciudad",
    "Propuesta para la instancia ciudad",
)

#### Propuesta para la instancia ciudad

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


| Lote | Tipo | Estación | Costo (millones) | Casos que cubre |
|---|---|---|---|---|
| L01 | accidente | Estación de tránsito y grúas | 2600 | 234 |
| L04 | hurto | CAI | 850 | 355 |
| L08 | hurto | CAI | 1000 | 240 |
| L11 | hurto | CAI | 1000 | 261 |
| L14 | hurto | CAI | 1100 | 84 |
| L19 | accidente | Estación de tránsito y grúas | 1600 | 233 |
| L25 | accidente | Estación de tránsito y grúas | 1700 | 89 |
| L26 | accidente | Estación de tránsito y grúas | 2800 | 153 |
| L28 | incendio | Estación de bomberos | 4000 | 74 |
| L29 | incendio | Estación de bomberos | 4250 | 64 |
| L32 | accidente | Estación de tránsito y grúas | 2100 | 377 |
| L34 | hurto | CAI | 1400 | 202 |
| L35 | hurto | CAI | 800 | 160 |
| L37 | accidente | Estación de tránsito y grúas | 1600 | 83 |
| L39 | accidente | Estación de tránsito y grúas | 2300 | 303 |
| L45 | accidente | Estación de tránsito y grúas | 1600 | 92 |
| L46 | hurto | CAI | 1150 | 254 |
| L48 | hurto | CAI | 1400 | 111 |
| L57 | accidente | Estación de tránsito y grúas | 1600 | 159 |
| L62 | hurto | CAI | 800 | 173 |
| L67 | accidente | Estación de tránsito y grúas | 1600 | 186 |
| L69 | accidente | Estación de tránsito y grúas | 1900 | 261 |
| L70 | hurto | CAI | 800 | 299 |

- Costo total: 39950 millones, frente a un presupuesto de 40000 millones.
- Casos cubiertos: 1887 hurtos, 1718 accidentes y 133 incendios.
- Valor de la propuesta: 7706.

**Justificación.** Para la instancia ciudad se recomienda la propuesta de la heurística de eficiencia: alcanza un valor de 7706, el más alto entre las alternativas obtenidas, con un costo de 39950 millones sobre un presupuesto de 40000 (99.9 % utilizado), y cubre 1887 hurtos, 1718 accidentes y 133 incendios. Frente a la propuesta de la heurística base (valor 7591) suma 115 puntos, con esta diferencia en casos cubiertos: hurto +537, accidente -39, incendio -61. No se garantiza que sea la mejor propuesta posible: en esta instancia el espacio de búsqueda es demasiado grande para recorrerlo por completo.

In [24]:
instancia = cargar_instancia("comuna")
alcances = calcular_alcances(instancia)
resumenes = {
    etiqueta: resultados_heuristicas[("comuna", etiqueta)][0] for etiqueta in HEURISTICAS
}
resumenes["optima"] = optimos["comuna"][0]

presentar_propuesta(
    instancia, alcances, resumenes, "optima", True, "comuna_optima",
    "Mejor propuesta posible para la instancia comuna",
)

#### Mejor propuesta posible para la instancia comuna

| Lote | Tipo | Estación | Costo (millones) | Casos que cubre |
|---|---|---|---|---|
| L03 | accidente | Estación de tránsito y grúas | 1800 | 222 |
| L06 | hurto | CAI | 850 | 180 |
| L09 | hurto | CAI | 850 | 166 |
| L11 | hurto | CAI | 1050 | 200 |
| L13 | accidente | Estación de tránsito y grúas | 2100 | 213 |
| L14 | hurto | CAI | 800 | 159 |
| L15 | incendio | Estación de bomberos | 4250 | 31 |

- Costo total: 11700 millones, frente a un presupuesto de 12000 millones.
- Casos cubiertos: 519 hurtos, 300 accidentes y 31 incendios.
- Valor de la propuesta: 1574.

**Justificación.** Para la instancia comuna se recomienda la mejor propuesta posible: alcanza un valor de 1574, el más alto entre las alternativas obtenidas, con un costo de 11700 millones sobre un presupuesto de 12000 (97.5 % utilizado), y cubre 519 hurtos, 300 accidentes y 31 incendios. Frente a la propuesta de la heurística base (valor 1529) suma 45 puntos, con esta diferencia en casos cubiertos: hurto +33, accidente +4, incendio +0. Frente a la propuesta de la heurística de eficiencia (valor 1492) suma 82 puntos, con esta diferencia en casos cubiertos: hurto -13, accidente -20, incendio +31. Además, la búsqueda exacta garantiza que ninguna propuesta que respete el presupuesto alcanza un valor mayor.